# Yield model (SPM + Random Forest) — Solapur (sorghum (jowar))

Sums APAR per growth stage and per season, builds the SPM yield proxy (ΣAPAR × RUE × HI), samples features at CCE points, trains a Random Forest (1000 trees) and predicts a 10 m yield map.

**Inputs:** `output/` mosaics, CCE yield table (`cce/`)  
**Outputs:** `output/<District>_<crop>_yield.tif` (kg/ha), `Regplot.png`, points workbook  
**Run after:** `02_raster_formation/raster_formation_solapur`  
**Next:** `05_accuracy_assessment/accuracy_solapur`  

> Update the path variables in the first cells before running. See [`docs/`](../../docs/) for methodology and parameters.
>
> ⚠️ `spm_processing()` writes `Yield.tif`, but the feature list expects `Yield1.tif` (Belagavi: `Yield3.tif`). Also, the model is fitted on all points, so the metrics are in-sample. See `docs/known-issues.md`.

In [ ]:
import os
import shutil
import rasterio as rio
from rasterio.merge import merge
import geopandas as gpd
import numpy as np
import pandas as pd
from datetime import datetime
import time

In [ ]:
import seaborn as sns

In [ ]:
from rasterio.crs import CRS

In [ ]:
# /// Growth stages
g0 = datetime.strptime('2022-11-01','%Y-%m-%d')
g1 = datetime.strptime('2022-12-01','%Y-%m-%d')
g2 = datetime.strptime('2022-12-31','%Y-%m-%d')
g3 = datetime.strptime('2023-01-30','%Y-%m-%d')
g4 = datetime.strptime('2023-02-28','%Y-%m-%d')

In [ ]:
base=r'C:\Local\Desktop_previous\miscellaneous\Neha\YieldData\21Jan\solapur'
outpath=os.path.join(base,'output')
if not os.path.exists(outpath):
    os.mkdir(outpath)

In [ ]:
##apar processing
apar_loc=outpath
image=[]
date=[]
for r,d,f in os.walk(apar_loc):
    for fl in f:
        if 'APAR' in fl and fl.endswith('.tif'):
            date.append(fl.split('-')[-1].split('.')[0])
            image.append(os.path.join(r,fl))
            
apar_df=pd.DataFrame({'AP':image,'Date':date})

apar_df['Date']=pd.to_datetime(apar_df['Date'],format='%Y_%m_%d')

In [ ]:
def apar_processing(imageList,stage):
    imgStack=[]
    for d in imageList:
        img=rio.open(d)
        data=img.read(1)
        imgStack.append(data)
    c=0
    for k in imgStack:
        k=k/1000
        if c==0:
            ar_sum=k
            c+=1
        else:
            ar_sum=np.add(ar_sum,k)
            
            print(ar_sum.shape)
    kwargs = img.meta
    kwargs.update(
        dtype=rio.float32,
        count=1,
#         crs = CRS({"init": "epsg:4326"}),
        compress='lzw')
    fname='Sum_apar_'+str(stage)
    with rio.open(os.path.join(outpath, fname+'.tif'), 'w', **kwargs) as dst:
        dst.write_band(1, ar_sum.astype(rio.float32))
    return fname

In [ ]:
def spm_processing(imageList,hi,rue):
    imgStack=[]
    for d in imageList:
        img=rio.open(d)
        data=img.read(1)
        imgStack.append(data)
    sumImage=sum(imgStack)*hi*rue
    ndv=(sumImage).min()
    sumImage=np.where(sumImage==ndv,np.nan,sumImage)
    kwargs = img.meta
    kwargs.update(
        dtype=rio.float32,
        count=1,
        compress='lzw')
    fname='Yield'
    with rio.open(os.path.join(outpath, fname+'.tif'), 'w', **kwargs) as dst:
        dst.write_band(1, sumImage.astype(rio.float32))
    return fname

In [ ]:
def spm_processing1(image,hi,rue):
    img=rio.open(image)
    data=img.read(1)
    sumImage=data*hi*rue
    ndv=(sumImage).min()
    sumImage=np.where(sumImage==ndv,np.nan,sumImage)
    kwargs = img.meta
    kwargs.update(
        dtype=rio.float32,
        count=1,
        compress='lzw')
    fname='Yield'
    with rio.open(os.path.join(outpath, fname+'.tif'), 'w', **kwargs) as dst:
        dst.write_band(1, sumImage.astype(rio.float32))
    return fname

In [ ]:
def stressFactor(image,ws,ts):
    img=rio.open(image)
    img_data=img.read(1)
    w=rio.open(ws)
    ws_data=w.read(1)/1000
    t=rio.open(ts)
    ts_data=t.read(1)/1000
    yld=img_data*ws_data*ts_data
    kwargs = img.meta
    kwargs.update(
        dtype=rio.float32,
        count=1,
        compress='lzw')
    fname='Yield_ts_ws'
    with rio.open(os.path.join(outpath, fname+'.tif'), 'w', **kwargs) as dst:
        dst.write_band(1, yld.astype(rio.float32))
    return fname

In [ ]:
st1=apar_df[(apar_df['Date']>g0) & (apar_df['Date']<=g1)]
st2=apar_df[(apar_df['Date']>g1) & (apar_df['Date']<=g2)]
st3=apar_df[(apar_df['Date']>g2) & (apar_df['Date']<=g3)]
st4=apar_df[(apar_df['Date']>g3)]

In [ ]:
apar_processing(list(st1['AP']),1)
apar_processing(list(st2['AP']),2)
apar_processing(list(st3['AP']),3)
apar_processing(list(st4['AP']),4)

In [ ]:
stg3=os.path.join(outpath,'Sum_apar_3.tif')
ws=os.path.join(base,'WS.tif')
ts=os.path.join(base,'TS.tif')
# stressFactor(stg3,ws,ts)

In [ ]:
apars=[]
for r,d,f in os.walk(outpath):
    for fl in f:
        if 'Sum_apar' in fl:
            apars.append(os.path.join(r,fl))

In [ ]:
apars

In [ ]:
apar_processing(apars,5)
apar_fin=[os.path.join(outpath,'Sum_apar_5.tif')]

In [ ]:
spm_processing(apar_fin,2.8,0.6)
# spm_processing1(apar_fin[0],4.65,0.26)

In [ ]:
# Final shape file  - CCE#

shapefile=pd.read_excel(os.path.join(base,'cce','V_Solapur_sorghum_yield.xlsx'))
# shapefile['Id']=np.arange(1,shapefile.shape[0]+1)
coords = [(x,y) for x, y in zip(shapefile.Longitude, shapefile.Latitude)]

In [ ]:
## location of bands from where the data is to be extracted
band_loc=outpath

In [ ]:
def getRasterValue(image,coords):
    ras = rio.open(image)
    return [x[0] for x in ras.sample(coords)]

In [ ]:
# X_bands = ['EVI1', 'NDPI1', 'NDVI1', 'SAVI1', 'NDWI1', 'NDTI1', 'EVI2', 'NDPI2',
#        'NDVI2', 'SAVI2', 'NDWI2', 'NDTI2', 'EVI3', 'NDPI3', 'NDVI3', 'SAVI3',
#        'NDWI3', 'NDTI3', 'EVI4', 'NDPI4', 'NDVI4', 'SAVI4', 'NDWI4', 'NDTI4', 'Yield1']
X_bands = ['EVI4', 'NDPI4', 'NDVI4', 'SAVI4', 'NDWI4', 'NDTI4', 'Yield1']

In [ ]:
for d in X_bands:
    shapefile[d]=getRasterValue(os.path.join(band_loc,d+'.tif'),coords)

In [ ]:
# shapefile['Yield(kg_p_ha)']=shapefile['Wet_Weight']*400

In [ ]:
df=shapefile[(~shapefile['Yield1'].isna()) & (shapefile['Yield1']>=0)]
# df['Yield']=df['Yield']*10000
# df=df[df['Yield']>3000]

In [ ]:
### generating models

from sklearn.ensemble import RandomForestRegressor

regressor = RandomForestRegressor(n_estimators=1000,random_state=123, criterion='friedman_mse')


In [ ]:
###  test train data

from sklearn.model_selection import train_test_split

In [ ]:
df.columns

In [ ]:
# df['Yield(kg_p_ha)']=df['Yield (kg/ha)']

In [ ]:
X = np.array(df[X_bands])
y = np.array(df[['Yield (kg_p_ha)']])

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state=123)

In [ ]:
regressor.fit(X, y)

In [ ]:
X_data=np.array(df[X_bands])

In [ ]:
y_predcted_rf= regressor.predict(X_data)

In [ ]:
yld_rf=pd.DataFrame(y_predcted_rf)
df['Predicted(rf)']=y_predcted_rf

In [ ]:
sns.regplot(data=df,x='Yield (kg_p_ha)',y='Predicted(rf)')

In [ ]:
# pip install seaborn

In [ ]:
import seaborn as sns
# from osgeo import gdal

In [ ]:
def rasterData(image):
    ras = rio.open(image)
    vec = ras.read(1)
    return vec.flatten()

In [ ]:
def yieldWriter(arr):
    img= rio.open(os.path.join(band_loc,"EVI1.tif"))
    img_data=img.read(1)
    Y_im_dt = arr.reshape(img_data.shape)
    kwargs = img.meta
    kwargs.update(
        dtype=rio.float32,
        count=1,
        compress='lzw')
    fname='Yield_predicted_new'
    with rio.open(os.path.join(outpath, fname+'.tif'), 'w', **kwargs) as dst:
        dst.write_band(1, Y_im_dt.astype(rio.float32))
    return fname

In [ ]:
vec=[]
for d in X_bands:
    fn=os.path.join(band_loc,d+'.tif')
    x=(rasterData(fn))
    ndv=x.min()
    x=np.where(x==ndv,np.nan,x)
    vec.append(x)

In [ ]:
X_vec=np.stack(vec).T

In [ ]:
from numpy import *
where_are_NaNs = isnan(X_vec)
X_vec[where_are_NaNs] = 0

In [ ]:
yield_predcted_rf= regressor.predict(X_vec)

In [ ]:
yieldWriter(yield_predcted_rf)

In [ ]:
def yieldMapRefined(image):
    img= rio.open(image)
    img_data=img.read(1)
    
    ndv=721.183777
    img_data=np.where(img_data==ndv,np.nan,img_data)
    kwargs = img.meta
    kwargs.update(
        dtype=rio.float32,
        count=1,
        compress='lzw')
    fname='Solapur_sorghum_yield_new_one'
    with rio.open(os.path.join(outpath, fname+'.tif'), 'w', **kwargs) as dst:
        dst.write_band(1, img_data.astype(rio.float32))
    return fname

In [ ]:
yieldMapRefined(os.path.join(outpath, 'Yield_predicted_new.tif'))

In [ ]:
fn=os.path.join(outpath, 'Solapur_sorghum_yield_new_one.tif')
# fn = r"C:\Users\pushkargaur\Dropbox (IFPRI)\Documents\My docs\Desktop_previous\miscellaneous\Neha\YieldData\raw\Siddipet_Maize_Rabi2022-23\output\Siddipet_Maize_yield.tif"

In [ ]:
shapefile['Predicted']=getRasterValue(fn,coords)

In [ ]:
df=shapefile[(~shapefile['Predicted'].isna()) & (shapefile['Predicted']>0)]
# df['Yield(kg_p_ha)']=df['Yield (kg/ha)']

In [ ]:
df['difference']=abs(df['Predicted']-df['Yield (kg_p_ha)'])

In [ ]:
df.shape[0]

In [ ]:
# df=df[df['difference']<2000]

In [ ]:
reg_plot=sns.regplot(data=df,x='Yield (kg_p_ha)',y='Predicted')
reg_fig=reg_plot.get_figure()
fn=os.path.join(outpath,'Regplot.png')
reg_fig.savefig(fn)

In [ ]:
def mape(y_true, y_pred): 
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    return np.mean(np.abs((y_true - y_pred) / y_true)) * 100


In [ ]:
from sklearn.metrics import mean_absolute_error,mean_squared_error,r2_score
y_test=np.array(df['Yield (kg_p_ha)'])
y_pred=np.array(df['Predicted'])
mae = mean_absolute_error(y_true=y_test,y_pred=y_pred)
# squared True returns MSE value, False returns RMSE value.
mse = mean_squared_error(y_true=y_test,y_pred=y_pred) #default=True
rmse = mean_squared_error(y_true=y_test,y_pred=y_pred,squared=False)
r2 = r2_score(y_true=y_test,y_pred=y_pred)
maper = mape(y_test,y_pred)

print("MAE:",round(mae,2))
print("MSE:",round(mse,2))
print("RMSE:",round(rmse,2))
print("R2:",round(r2,2))
print("MAPE:",round(maper,2))


In [ ]:
df.to_excel(os.path.join(outpath,'Solapur_sorghum_yield_stats.xlsx'))

In [ ]:
df